In [ ]:
import sys
import os
import torch
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.notebook import tqdm
import medmnist
from medmnist import INFO

# Set Plot Style for Video Presentation
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300 # High resolution for video
plt.rcParams['font.family'] = 'sans-serif'

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.models.raresight_net import RareSight

# Device Config
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running evaluation on: {device}")

# Load Class Names
CLASSES = {
    0: 'Actinic keratoses', 1: 'Basal cell carcinoma', 2: 'Benign keratosis',
    3: 'Dermatofibroma', 4: 'Melanoma', 5: 'Melanocytic nevi', 6: 'Vascular lesions'
}

In [ ]:
# Initialize Model
model = RareSight(device=device)

# Load Weights
weight_path = '../checkpoints/raresight_best.pth'

if os.path.exists(weight_path):
    state_dict = torch.load(weight_path, map_location=device)
    model.load_state_dict(state_dict)
    print("✅ Trained weights loaded successfully.")
    print(f"Learned Alpha Value: {model.alpha.item():.4f}")
else:
    print("⚠️ Warning: Weights not found. Using random initialization.")

model.eval()

In [ ]:
# --- CONFIGURATION ---
N_WAY = 5
K_SHOT = 5
N_QUERY = 15
N_EPISODES = 600 # Standard evaluation count

# Load TEST Data (Not Train!)
info = INFO['dermamnist']
DataClass = getattr(medmnist, info['python_class'])
test_dataset = DataClass(split='test', download=True)

# Load Descriptions
with open('../src/app/class_descriptions.json', 'r') as f:
    DESCRIPTIONS = json.load(f)

# Storage for Metrics
all_preds = []
all_targets = []
feature_vectors = [] # For t-SNE
feature_labels = []

print(f"Starting evaluation on {N_EPISODES} episodes...")

with torch.no_grad():
    for _ in tqdm(range(N_EPISODES)):
        # 1. Sample Episode
        classes = np.unique(test_dataset.labels)
        selected_classes = np.random.choice(classes, N_WAY, replace=False)
        
        s_imgs, s_txts = [], []
        q_imgs = []
        
        # Local (Episode) labels and Global (Dataset) labels
        q_global_labels = [] 

        for i, cls in enumerate(selected_classes):
            # Find indices for this class
            indices = np.where(test_dataset.labels == cls)[0]
            
            # Safety check for small classes in test set
            needed = K_SHOT + N_QUERY
            replace = len(indices) < needed
            chosen = np.random.choice(indices, needed, replace=replace)
            
            # Support
            desc = DESCRIPTIONS.get(str(cls), "Skin lesion")
            for idx in chosen[:K_SHOT]:
                img, _ = test_dataset[idx]
                s_imgs.append(model.preprocess(img).unsqueeze(0))
                s_txts.append(desc)
                
            # Query
            for idx in chosen[K_SHOT:]:
                img, _ = test_dataset[idx]
                q_imgs.append(model.preprocess(img).unsqueeze(0))
                q_global_labels.append(cls) # Store global label (0-6)
                
        # 2. Prepare Tensors
        s_tensor = torch.cat(s_imgs).to(device)
        q_tensor = torch.cat(q_imgs).to(device)
        
        # 3. Forward Pass
        # Returns logits: [N_Query_Total, N_Way]
        logits = model(s_tensor, s_txts, q_tensor, N_WAY, K_SHOT)
        
        # 4. Predictions
        # Argmax gives index 0..4 (relative to episode)
        probs = torch.softmax(logits, dim=1)
        preds_local = torch.argmax(probs, dim=1).cpu().numpy()
        
        # Map Local Preds back to Global Labels
        # selected_classes = [3, 0, 4, ...] -> preds_local 0 means class 3
        preds_global = [selected_classes[p] for p in preds_local]
        
        # Store Data
        all_preds.extend(preds_global)
        all_targets.extend(q_global_labels)
        
        # Store embeddings for t-SNE (Only first 100 episodes to save RAM)
        if len(feature_vectors) < 1000:
            # Extract query features directly from backbone for visualization
            feats = model.backbone.encode_image(q_tensor)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            feature_vectors.extend(feats.cpu().numpy())
            feature_labels.extend(q_global_labels)

# Calculate Accuracy
acc = np.mean(np.array(all_preds) == np.array(all_targets))
print(f"\n🏆 Overall Few-Shot Test Accuracy: {acc*100:.2f}%")

In [ ]:
# Compute Matrix
cm = confusion_matrix(all_targets, all_preds, normalize='true')

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=CLASSES.values(), yticklabels=CLASSES.values())

plt.title(f'RareSight Confusion Matrix (Acc: {acc*100:.1f}%)', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Diagnosis', fontsize=12)
plt.ylabel('True Diagnosis', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate Per-Class Accuracy
report = classification_report(all_targets, all_preds, output_dict=True, zero_division=0)

class_accs = []
class_names = []
colors = []

for cls_id, name in CLASSES.items():
    if str(cls_id) in report:
        acc_score = report[str(cls_id)]['recall'] # Recall = Accuracy per class
        class_accs.append(acc_score * 100)
        class_names.append(name)
        
        # Highlight Dermatofibroma (Class 3) and Vascular (Class 6)
        if cls_id in [3, 6]:
            colors.append('#FF6B6B') # Red for Rare
        else:
            colors.append('#4ECDC4') # Teal for Common

# Plot
plt.figure(figsize=(12, 6))
bars = plt.bar(class_names, class_accs, color=colors, edgecolor='black', alpha=0.9)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Per-Class Diagnostic Accuracy', fontsize=16, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12)
plt.axhline(y=acc*100, color='gray', linestyle='--', alpha=0.7, label='Mean Accuracy')
plt.legend(['Mean Accuracy', 'Common Disease', 'Rare Disease'])
plt.xticks(rotation=30, ha='right')
plt.ylim(0, 100)

# Legend Hack
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4ECDC4', edgecolor='black', label='Common Disease'),
                   Patch(facecolor='#FF6B6B', edgecolor='black', label='Rare Disease (Target)')]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
print("Computing t-SNE projection... (this may take a moment)")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_embedded = tsne.fit_transform(np.array(feature_vectors))

# Create DataFrame for Plotting
df_tsne = pd.DataFrame(X_embedded, columns=['x', 'y'])
df_tsne['label'] = [CLASSES[l] for l in feature_labels]

# Plot
plt.figure(figsize=(12, 10))
sns.scatterplot(data=df_tsne, x='x', y='y', hue='label', palette='tab10', s=60, alpha=0.8)

plt.title('t-SNE Projection of RareSight Feature Space', fontsize=16, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Diagnosis')
plt.tight_layout()
plt.axis('off') # Hide axes for cleaner look
plt.show()

In [ ]:
# Hardcoded Baseline Data (From your Prompt)
models = ['ResNet-50 (Transfer)', 'ProtoNet (Image-Only)', 'BiomedCLIP (Zero-Shot)', 'RareSight (Ours)']
accuracies = [78.35, 60.99, 41.99, acc*100] 
# Note: If your trained model performs differently, the bar will adjust automatically.

plt.figure(figsize=(10, 6))
bars = plt.barh(models, accuracies, color=['gray', 'gray', 'gray', '#2E86C1'])

# Highlight ours
bars[-1].set_color('#2E86C1')
bars[-1].set_edgecolor('black')
bars[-1].set_linewidth(2)

for i, v in enumerate(accuracies):
    plt.text(v + 1, i, f'{v:.2f}%', va='center', fontweight='bold', fontsize=12)

plt.title('Midpoint Evaluation: RareSight vs Baselines', fontsize=16, fontweight='bold')
plt.xlabel('Test Accuracy (%)', fontsize=12)
plt.xlim(0, 100)
plt.tight_layout()
plt.show()